# Pandas API on Spark

The **pandas API on Spark** (`pyspark.pandas`) lets you use familiar pandas
syntax on top of Apache Spark, combining pandas ease-of-use with Spark scalability.

This notebook covers:
1. Object creation (Series, DataFrame)
2. Basic operations (head, describe, sort, transpose)
3. Selection and filtering
4. Missing data handling
5. Aggregation and grouping
6. Conversion between pandas, Spark, and pandas-on-Spark

## Setup

In [ ]:
import os
os.environ['PYARROW_IGNORE_TIMEZONE'] = '1'

import numpy as np
import pandas as pd
import pyspark.pandas as ps
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('pandas-on-spark-notebook')
    .master(os.environ.get('SPARK_MASTER', 'local[*]'))
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
    .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version}')

## 1. Object Creation

### Series

In [ ]:
pss = ps.Series([1, 3, 5, np.nan, 6, 8], name='values')
pss

### DataFrame from dict

In [ ]:
psdf = ps.DataFrame({
    'name':  ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank'],
    'age':   [30, 25, 35, 28, 32, 45],
    'score': [85.5, 92.0, 78.0, 88.5, 95.0, 70.5],
    'city':  ['NYC', 'LA', 'NYC', 'LA', 'NYC', 'LA'],
})
psdf

### DataFrame from pandas

In [ ]:
pdf = pd.DataFrame(np.random.default_rng(42).standard_normal((5, 3)),
                   columns=['A', 'B', 'C'])
psdf_from_pd = ps.from_pandas(pdf)
psdf_from_pd

## 2. Basic Operations

In [ ]:
print('--- head ---')
psdf.head(3)

In [ ]:
print('--- describe ---')
psdf.describe()

In [ ]:
print('--- sort by score descending ---')
psdf.sort_values('score', ascending=False)

In [ ]:
print('--- dtypes ---')
psdf.dtypes

## 3. Selection & Filtering

In [ ]:
# Column selection
psdf[['name', 'score']]

In [ ]:
# Boolean filtering
psdf[psdf['score'] > 85]

In [ ]:
# isin filtering
psdf[psdf['city'].isin(['NYC'])]

## 4. Missing Data

In [ ]:
df_with_na = ps.DataFrame({
    'A': [1, np.nan, 3, np.nan, 5],
    'B': [10, 20, np.nan, 40, 50],
})
print('--- original ---')
print(df_with_na)

print('\n--- dropna ---')
print(df_with_na.dropna())

print('\n--- fillna(0) ---')
print(df_with_na.fillna(0))

## 5. Aggregation & Grouping

In [ ]:
psdf.groupby('city').agg({'score': ['mean', 'max'], 'age': 'mean'})

In [ ]:
# Value counts
psdf['city'].value_counts()

## 6. Apply & Transform

In [ ]:
# Apply a function to a column
psdf['grade'] = psdf['score'].apply(lambda s: 'A' if s >= 90 else 'B' if s >= 80 else 'C')
psdf[['name', 'score', 'grade']]

## 7. Conversion

Three-way conversion between pandas, Spark, and pandas-on-Spark.

In [ ]:
# pandas-on-Spark → Spark DataFrame
sdf = psdf.to_spark()
sdf.show(3)

In [ ]:
# Spark DataFrame → pandas-on-Spark
psdf_back = sdf.pandas_api()
psdf_back.head(3)

In [ ]:
# pandas-on-Spark → pandas (driver memory)
local_pdf = psdf.to_pandas()
print(type(local_pdf))
local_pdf.head(3)

In [ ]:
spark.stop()